# Complex-Time Anharmonic Oscillator — MU rotation contour

Rebuild of the NN contour model along the lines of `ND_integrals_MU.ipynb`.
The previous `x = A^T z + b + g(z)` version is preserved in `Complex_Time_QHO_Anharmonic_MU.ipynb`.

## Model

The linear-map-plus-correction model is replaced by a **pure rotation of the DST modes**:

$$x \;=\; Q\Bigl[(s\odot z)\odot e^{\,i\theta(z)}\Bigr],
\qquad s_k = |d_k|^{-1/2}$$

- One MLP: three hidden layers of width 64, `Tanh`, and $D$ real outputs $\theta$.
- $x = 0$ at $z = 0$, so the contour passes through the saddle (exact for zero endpoints).
- The final layer is initialised at ~0 so $\theta \approx 0$. The run starts on a
  **neutral real-axis contour, not at the $\lambda=0$ thimble** — it has to find the
  rotation itself, so independent seeds become a real test of whether the loss determines
  the contour.
- The rotation acts in the **mode** frame, where the harmonic action is diagonal. The exact
  $\lambda=0$ thimble is $\theta_k = \tfrac12(\pi/2 - \arg d_k) \approx 0.7576$, so it lies
  *inside* this model class (the spread across modes is only 0.006 rad).
- $Q$ and $s$ are **fixed and real**, built from $|d_k|$ only, so they carry **no phase
  information**. They are a preconditioner, not an answer: they put the mode widths on a
  physical scale, which matters because those widths span a factor of ~21.

### Why magnitudes are held fixed

`ALLOW_MAGNITUDE = False` is load-bearing, not cosmetic. If the network is also allowed to
emit a log-magnitude $\rho$, the loss below has a **degenerate global minimum at $x \to 0$**:
the contour collapses onto the saddle, where $S \equiv 0$ and the phase condition is
satisfied for free. This was measured — $\mathrm{Re}\,S$ fell from 15.5 to $-0.013$ (looks
like success) while $|S|$ collapsed from 15.5 to 0.21, $\log\det J \to -\infty$ and $K$
underflowed to $0$. Fixing $|s_k z_k|$ closes that channel: the contour can only rotate, so
it cannot reach the saddle and cannot escape to where the integrand underflows.

## Loss

Variance of $\log W$ is gone. Training uses the scaled phase condition plus the descent
condition on the action alone — **no Jacobian, no sampling density**:

```python
scale        = torch.abs(S).clamp_min(1e-12)
phase_loss   = torch.mean((S.real / scale)**2)
descent_loss = torch.mean(torch.relu(-S.imag)**2)
loss         = phase_loss + descent_loss
```

Gradients flow through `scale`. Adam, `lr = 1e-3`, batch 512, fresh standard normals every
step, trained from scratch. The two terms are logged and plotted separately.

Rationale: with weight $e^{iS}$, constant phase means $\mathrm{Im}(iS)=\mathrm{Re}(S)$ is
constant, and that constant is $0$ because $S=0$ at the $x=0$ saddle for zero endpoints.
Descent means $\mathrm{Re}(iS) = -\mathrm{Im}(S) \le 0$, i.e. $\mathrm{Im}(S) \ge 0$.

The descent term is what selects the *right* rotation. At $\lambda=0$ both
$\theta \approx 0.7576$ and $\theta \approx 0.7576 + \pi/2$ give $\mathrm{Re}\,S = 0$, but the
second has $\mathrm{Im}\,S < 0$ — an anti-thimble, exponentially growing instead of decaying.
`descent_loss` rejects it. Because $\eta > 0$ already gives the real-axis contour
$\mathrm{Im}\,S > 0$, this term is inactive at the start and only bites if training
overshoots; `<Im S>` is logged so you can see whether it ever engages.

## Estimation

Unchanged in form: $\log W = iS + \log\det J - \log P$, with the **full complex Jacobian**
including cross-coordinate derivatives, and the original $(m/2\pi i a_c)^{N/2}$ normalisation
and complex-time action.

$$J = Q\,\mathrm{diag}(e^{w})\,M,\qquad
M = \mathrm{diag}(s) + \mathrm{diag}(s\odot z)\,W,\qquad
W_{kl} = \frac{\partial w_k}{\partial z_l}$$

$$\log\det J = \log\det Q + \sum_k w_k + \log\det M$$

$W$ is the full $D\times D$ network Jacobian, obtained by $D$ exact reverse-mode passes
(samples are independent, so one pass yields a whole row for the batch). Holding $\theta$
directly makes $\sum_k w_k$ branch-free, which removes the $\pm\pi$ ambiguity that produced
the sign-flipped $\lambda=0$ result in the earlier notebook; only $\log\det M$ needs
`slogdet`, and $M$ starts at $\mathrm{diag}(s)$, real and positive. Because the Jacobian is
absent from the loss, $W$ is needed only at estimation time.

The preflight cell validates all of this: the analytic Jacobian against central differences,
and — more importantly — the whole estimator against the **exact** $\lambda=0$ thimble
$x = Q\,\mathrm{diag}(\sqrt{i/d_k})\,z$, which returns the reference to 5 decimal places at
$\mathrm{ESS}/N = 1.000000$. So if a trained contour disagrees, the contour is at fault, not
the estimator.

## Run order

$\lambda = 0$ first as a benchmark against the exact $N=32$ lattice Gaussian, then
$\lambda = 0.7$ and $\lambda = 2.4$.

In [1]:
import csv
import json
import os
import time
from datetime import datetime

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn

%matplotlib inline

# |iS| reaches O(10^3) here and gets exponentiated, so float32 is not enough.
torch.set_default_dtype(torch.float64)
torch.manual_seed(42)
np.random.seed(42)

print(f"torch {torch.__version__}  |  default dtype {torch.get_default_dtype()}")

torch 2.1.0+cpu  |  default dtype torch.float64


## 1. Parameters

Complex-time propagator $K(x_f, x_i; T_c)$ with $T_c = T - i\eta$ and complex step
$a_c = T_c / N$. $N = 32$ so the exact $N=32$ lattice references apply.

In [ ]:
# ── physics / discretisation ──
N       = 32  
T       = 1.0
eta     = 0.05
m       = 1.0
omega   = 1.0
D       = N - 1
T_c     = T - 1j * eta
a_c     = T_c / N

x_i_prop = 0.0
x_f_prop = 0.0
# phase_loss uses mean((Re S)^2), i.e. it targets the constant Re(S)=0 that holds at the
# x=0 saddle. That is only correct for zero endpoints; otherwise use var(Re S) instead.
assert x_i_prop == 0.0 and x_f_prop == 0.0, "phase_loss assumes S=0 at the saddle"

# ── model ──
WIDTH       = 64      # three hidden layers of width 64, Tanh
OUT_SCALE   = 1e-3    # final-layer std, so theta ~ 0 at init (neutral real-axis start)
USE_PRECOND = True    # fixed real DST mode widths |d_k|^-1/2; carries no phase information

# Pure rotation. Mode magnitudes s_k|z_k| are FIXED, which is what makes the phase loss
# well posed: Re(S)=0 cannot be satisfied by collapsing onto the saddle where S == 0.
# Setting this True lets the net also emit a log-magnitude rho. That was measured to
# collapse (Re S -> 0 with |S| -> 0, log det J -> -inf, K -> underflow), so keep it False
# unless you are deliberately reproducing that failure.
ALLOW_MAGNITUDE = False
 
# ── training ──
EPOCHS    = 10_000
BATCH     = 512
LR        = 1e-3
CLIP_GRAD = 5.0
LOG_EVERY = 1_000

# ── estimation ──
N_MC     = 100_000
MC_CHUNK = 5_000

# ── run plan ──
seed_list   = [52]        # independent random starts: seed agreement is the diagnostic
lambda_list = [0.0, 0.7, 2.4]     # lambda=0 benchmark first, then anharmonic

LAMBDA0_TOL            = 0.02     # 2% per component on the lambda=0 benchmark
ALLOW_GLOBAL_SIGN_FLIP = True     # det(B) sign is arbitrary; +K and -K both accepted
STOP_ON_LAMBDA0_FAIL   = False    # True -> abort before the anharmonic runs

# ── exact references ──
# Compare fixed-N ML against the N=32 LATTICE values; continuum is a separate limit.
LATTICE_REFERENCE_N32 = {
    0.0:  0.3121159 - 0.3022604j,
    0.5:  0.3057603 - 0.3077905j,
    0.7:  0.2684413 - 0.2878695j,
    2.4: -0.1109040 - 0.1529262j,
}
CONTINUUM_REFERENCE = {
    0.0:  0.3121370 - 0.3022849j,
    0.5:  0.3063511 - 0.3082963j,
    0.7:  0.2684401 - 0.2897428j,
    2.4: -0.1118026 - 0.1577383j,
}

# ── run directory ──
run_timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
run_tag = f"MU_rotor_omega{omega}_T{T}_eta{eta}_N{N}"
run_dir = os.path.join("runs", f"{run_timestamp}_{run_tag}")
os.makedirs(run_dir, exist_ok=True)

print(f"N={N}, T={T}, eta={eta}, T_c={T_c}, a_c={a_c}")
print(f"D={D}, width={WIDTH}, batch={BATCH}, lr={LR}, epochs={EPOCHS}, N_mc={N_MC}")
print(f"lambda_list={lambda_list}, seeds={seed_list}, endpoints=({x_i_prop}, {x_f_prop})")
print(f"preconditioner={USE_PRECOND}, out_scale={OUT_SCALE}, "
      f"allow_magnitude={ALLOW_MAGNITUDE}")
print(f"run_dir={run_dir}")

N=5, T=1.0, eta=0.05, T_c=(1-0.05j), a_c=(0.2-0.01j)
D=4, width=64, batch=512, lr=0.001, epochs=10000, N_mc=100000
lambda_list=[0.0, 0.7, 2.4], seeds=[52], endpoints=(0.0, 0.0)
preconditioner=True, out_scale=0.001, allow_magnitude=False
run_dir=runs\20260909_184023_MU_rotor_omega1.0_T1.0_eta0.05_N5


## 2. Action (complex time) — unchanged

$$S[x] \;=\; \sum_{k=0}^{N-1}\frac{m}{2a_c}(x_{k+1}-x_k)^2
\;-\; \frac{a_c m \omega^2}{2}\sum_{k=1}^{N-1} x_k^2
\;-\; \frac{a_c\lambda}{4}\sum_{k=1}^{N-1} x_k^4$$

Identical to the previous notebook apart from accumulating in `complex128`.
Endpoints are pinned, so only the $D = N-1$ interior points are free and $S=0$ at $x=0$.

In [3]:
def complex_action(x, x_i, x_f, lam_val, a_c_val, m_val, omega_val):
    """Discretised action with complex time step a_c = T_c/N. Weight is exp(iS)."""
    x = x.to(torch.complex128)
    bsz = x.shape[0]
    pad_i = torch.full((bsz, 1), complex(x_i), dtype=torch.complex128, device=x.device)
    pad_f = torch.full((bsz, 1), complex(x_f), dtype=torch.complex128, device=x.device)
    x_pad = torch.cat([pad_i, x, pad_f], dim=1)

    dx = x_pad[:, 1:] - x_pad[:, :-1]
    kinetic     = (m_val / (2 * a_c_val)) * torch.sum(dx**2, dim=1)
    pot_harm    = (a_c_val * m_val / 2) * omega_val**2 * torch.sum(x**2, dim=1)
    pot_quartic = a_c_val * (lam_val / 4) * torch.sum(x**4, dim=1)
    return kinetic - pot_harm - pot_quartic


# S must vanish at the saddle for phase_loss = mean((Re S)^2) to be the right target.
_S0 = complex_action(torch.zeros(1, D, dtype=torch.complex128),
                     x_i_prop, x_f_prop, 2.4, a_c, m, omega)
print(f"S at x=0 (lambda=2.4): {_S0.item():.3e}   <- must be 0 for the phase target")

S at x=0 (lambda=2.4): 0.000e+00+0.000e+00j   <- must be 0 for the phase target


## 3. Mode frame and widths (real, fixed, phase-free)

The harmonic Hessian of the interior points is tridiagonal and diagonalised by the real sine
(DST) basis $Q_{jk} = \sqrt{2/N}\,\sin(\pi jk/N)$, with eigenvalues

$$d_k \;=\; \frac{2m}{a_c} - a_c m\omega^2 - \frac{2m}{a_c}\cos\!\Bigl(\frac{\pi k}{N}\Bigr).$$

We take **only the magnitudes**, $s_k = |d_k|^{-1/2}$, so both $Q$ and $s$ are real. Two
printed numbers matter:

- the spread of $s_k$ shows how badly an unpreconditioned rotation (all widths $=1$) would be
  scaled — a factor of ~21 here, which is why the frame exists. With magnitudes held fixed
  this is the *only* chance to get the widths right, so it is not optional;
- the spread of $\arg d_k$ is ~0.006 rad, so the exact $\lambda=0$ thimble is a nearly
  *uniform* rotation $\theta_k \approx 0.7576$ and therefore lies inside the model class.
  **No phase is applied** — $\theta$ starts at ~0 and the network must find it.

In [4]:
def hessian_modes(N_val, a_c_val, m_val, omega_val):
    """DST basis Q and complex harmonic Hessian eigenvalues d_k for the interior points."""
    j = np.arange(1, N_val)[:, None]
    k = np.arange(1, N_val)[None, :]
    Q = np.sqrt(2.0 / N_val) * np.sin(np.pi * j * k / N_val)

    modes = np.arange(1, N_val)
    d = (2.0 * m_val / a_c_val
         - a_c_val * m_val * omega_val**2
         - 2.0 * (m_val / a_c_val) * np.cos(np.pi * modes / N_val))
    return Q, d


def mode_frame(N_val, a_c_val, m_val, omega_val, enabled=True):
    """Real orthogonal mode basis Q and real mode widths s_k = |d_k|^-1/2.

    Phase-free by construction: only |d_k| is used. Identity frame if disabled.
    """
    D_val = N_val - 1
    if not enabled:
        return (torch.eye(D_val, dtype=torch.float64),
                torch.ones(D_val, dtype=torch.float64), None)
    Q, d = hessian_modes(N_val, a_c_val, m_val, omega_val)
    widths = 1.0 / np.sqrt(np.abs(d))
    return (torch.tensor(Q, dtype=torch.float64),
            torch.tensor(widths, dtype=torch.float64), d)


Q_mode, s_mode, _d_modes = mode_frame(N, a_c, m, omega, enabled=USE_PRECOND)
assert torch.isreal(Q_mode).all() and torch.isreal(s_mode).all(), "frame must be real"
_orth = torch.max(torch.abs(Q_mode.T @ Q_mode - torch.eye(D))).item()
print(f"Q orthogonality residual: {_orth:.3e}   det(Q) = {np.linalg.det(Q_mode.numpy()):+.6f}")

if _d_modes is not None:
    _args = np.angle(_d_modes)
    THETA_EXACT_LAMBDA0 = 0.5 * (np.pi / 2 - _args)   # diagnostics only, never applied
    _w = s_mode.numpy()
    print(f"mode widths |d_k|^-1/2 : min={_w.min():.4f}  max={_w.max():.4f}  "
          f"ratio={_w.max()/_w.min():.1f}x   <- why a bare rotation is mis-scaled")
    print(f"arg(d_k)               : min={_args.min():.5f}  max={_args.max():.5f}  "
          f"spread={_args.max()-_args.min():.5f} rad")
    print(f"exact lambda=0 thimble phase: {THETA_EXACT_LAMBDA0.min():.4f} .. "
          f"{THETA_EXACT_LAMBDA0.max():.4f} rad  (pi/4 = {np.pi/4:.4f})")
    print("   ^ spread is tiny, so the exact lambda=0 thimble lies INSIDE this model class.")
    print("   ^ NOT applied at init: theta starts at ~0 and must be found by training.")
else:
    THETA_EXACT_LAMBDA0 = None

Q orthogonality residual: 1.477e-16   det(Q) = +1.000000
mode widths |d_k|^-1/2 : min=0.2366  max=0.7651  ratio=3.2x   <- why a bare rotation is mis-scaled
arg(d_k)               : min=0.05108  max=0.06165  spread=0.01058 rad
exact lambda=0 thimble phase: 0.7546 .. 0.7599 rad  (pi/4 = 0.7854)
   ^ spread is tiny, so the exact lambda=0 thimble lies INSIDE this model class.
   ^ NOT applied at init: theta starts at ~0 and must be found by training.


## 4. Rotation model and the full complex Jacobian

With mode amplitudes $u = s\odot z$ and exponent $w(z) = i\theta(z)$, the map is
$v_k = u_k e^{w_k}$ followed by $x = Qv$, so

$$\frac{\partial v_k}{\partial z_l}
= e^{w_k}\Bigl(s_k\delta_{kl} + u_k \frac{\partial w_k}{\partial z_l}\Bigr)
\;\;\Longrightarrow\;\;
J = Q\,\mathrm{diag}(e^{w})\,\underbrace{\bigl(\mathrm{diag}(s) + \mathrm{diag}(u)W\bigr)}_{M}$$

$$\log\det J = \log\det Q + \sum_k w_k + \log\det M$$

$W$ is the **full** $D\times D$ network Jacobian, so all cross-coordinate derivatives are
included. Notes:

- $\sum_k w_k$ is exact and branch-free because $\theta$ is held directly rather than
  recovered from a determinant. This removes the $\pm\pi$ global-sign ambiguity that produced
  the sign-flipped $\lambda=0$ result in the earlier notebook. Only $\det M$ needs `slogdet`,
  and $M$ starts at $\mathrm{diag}(s)$ — real, positive, far from any branch cut.
- $\det Q = \pm 1$ for an orthogonal frame, so $\log\det Q$ is $0$ or $i\pi$; it is computed
  once and contributes only a global sign.
- $W$ needs $D$ reverse-mode passes for the phase-only model ($2D$ if magnitudes are
  enabled). Samples are independent, so the gradient of `out[:, i].sum()` yields row $i$ of
  $W$ for the whole batch at once — exact, not a finite difference, and needed **only at
  estimation time** since the loss excludes $J$.

In [5]:
class ThimbleRotor(nn.Module):
    """x = Q [ (s * z) * exp(w(z)) ]   with   w = i*theta(z)   (pure rotation).

    The rotation acts on the DST modes, which is where the harmonic action is diagonal, so
    the exact lambda=0 thimble sits inside the model class. With ALLOW_MAGNITUDE=False the
    mode amplitudes |s_k z_k| are FIXED for every sample and every epoch. That is the whole
    point: the contour then cannot satisfy Re(S)=0 by collapsing onto the saddle (where
    S == 0 identically), and cannot drift out to where the integrand underflows.

    One MLP: three width-64 Tanh hidden layers, D real outputs (theta). x(0) = 0, so the
    contour passes through the saddle. The final layer starts at ~0, so theta ~ 0 and the
    run begins on the real axis rather than at the lambda=0 thimble.
    """

    def __init__(self, dim, Q, s, width=64, out_scale=1e-3, allow_magnitude=False):
        super().__init__()
        self.dim = dim
        self.allow_magnitude = allow_magnitude
        n_out = 2 * dim if allow_magnitude else dim

        self.register_buffer("Q", Q.clone().to(torch.float64))
        self.register_buffer("Qc", Q.clone().to(torch.complex128))
        self.register_buffer("s", s.clone().to(torch.float64))
        _sgn, _la = torch.linalg.slogdet(self.Q)        # det Q = +-1 for an orthogonal frame
        self.register_buffer(
            "log_det_Q", (_la + 1j * torch.angle(_sgn)).to(torch.complex128))

        self.net = nn.Sequential(
            nn.Linear(dim, width), nn.Tanh(),
            nn.Linear(width, width), nn.Tanh(),
            nn.Linear(width, width), nn.Tanh(),
            nn.Linear(width, n_out),
        )
        nn.init.normal_(self.net[-1].weight, std=out_scale)
        nn.init.zeros_(self.net[-1].bias)

    # ---- exponent w and its exact Jacobian -------------------------------------------
    def _w(self, z):
        out = self.net(z)
        if self.allow_magnitude:
            return (out[:, :self.dim].to(torch.complex128)
                    + 1j * out[:, self.dim:].to(torch.complex128))
        return 1j * out.to(torch.complex128)            # phase only

    def _W(self, z):
        """W[b, k, l] = d w_k / d z_l, exact reverse-mode, all cross terms included.

        Samples are independent, so grad(out[:, i].sum()) yields row i for the whole batch.
        Needed only at estimation time, since the loss does not involve the Jacobian.
        """
        with torch.enable_grad():
            zg = z.detach().clone().requires_grad_(True)
            out = self.net(zg)
            n_out = out.shape[1]
            rows = []
            for i in range(n_out):
                (grad_i,) = torch.autograd.grad(
                    out[:, i].sum(), zg, retain_graph=(i < n_out - 1), create_graph=False)
                rows.append(grad_i)
            G = torch.stack(rows, dim=1).detach()        # (b, n_out, dim), real
        if self.allow_magnitude:
            return G[:, :self.dim].to(torch.complex128) + 1j * G[:, self.dim:].to(torch.complex128)
        return 1j * G.to(torch.complex128)

    # ---- map and Jacobian ------------------------------------------------------------
    def forward(self, z):
        u = (z * self.s).to(torch.complex128)            # mode amplitudes
        w = self._w(z)
        v = u * torch.exp(w)                             # rotate each mode
        return v @ self.Qc.T, w                          # back to position space

    def _jac_parts(self, z):
        """J = Q diag(e^w) M  with  M = diag(s) + diag(s*z) W."""
        W = self._W(z)
        with torch.no_grad():
            u = (z * self.s).to(torch.complex128)
            w = self._w(z)
            M = (torch.diag(self.s).to(torch.complex128).unsqueeze(0)
                 + u.unsqueeze(-1) * W)
        return w, M

    def log_det_J(self, z):
        """log det J = log det Q + sum_k w_k + log det M. Detached; estimation only.

        The sum over w is exact and branch-free because we hold theta directly rather than
        recovering a phase from a determinant. Only log det M needs slogdet, and M starts
        at diag(s), i.e. real and positive.
        """
        w, M = self._jac_parts(z)
        with torch.no_grad():
            sign, logabs = torch.linalg.slogdet(M)
            return self.log_det_Q + w.sum(dim=1) + logabs + 1j * torch.angle(sign)

    def jacobian_matrix(self, z):
        """Exact D x D complex Jacobian per sample (folding diagnostics, preflight)."""
        w, M = self._jac_parts(z)
        with torch.no_grad():
            inner = torch.exp(w).unsqueeze(-1) * M
            return torch.matmul(self.Qc.unsqueeze(0), inner)


print(f"ThimbleRotor defined (allow_magnitude={ALLOW_MAGNITUDE}).")

ThimbleRotor defined (allow_magnitude=False).


## 5. Preflight checks

Cheap and worth running before any training:

1. the analytic Jacobian and `log det J` against central finite differences, on a small
   model with a deliberately non-symmetric $B$ and a *large* output scale so the nonlinear
   part actually contributes;
2. the closed-form $N=32$ harmonic lattice Gaussian against the tabulated $\lambda=0$
   reference, confirming that the reference this notebook is graded against is the right one.

In [6]:
def finite_difference_check(eps=1e-6, dim=5, tol=1e-7):
    """Validate jacobian_matrix and log_det_J against central differences.

    Uses a generic (non-identity) orthogonal frame, unequal mode widths and a LARGE output
    scale, so the nonlinear part of the Jacobian genuinely contributes. Both phase-only and
    magnitude-enabled variants are checked.
    """
    torch.manual_seed(123)
    Q_t, _ = torch.linalg.qr(torch.randn(dim, dim))
    s_t = torch.linspace(0.3, 2.5, dim)

    for allow_mag in (False, True):
        model = ThimbleRotor(dim, Q_t, s_t, width=16,
                             out_scale=0.35, allow_magnitude=allow_mag)
        z = torch.randn(4, dim)
        _fd_compare(model, z, eps, tol, allow_mag)


def _fd_compare(model, z, eps, tol, allow_mag):
    J_exact = model.jacobian_matrix(z)
    with torch.no_grad():
        J_fd = torch.empty_like(J_exact)
        for k in range(z.shape[1]):
            dz = torch.zeros_like(z)
            dz[:, k] = eps
            x_plus, _ = model(z + dz)
            x_minus, _ = model(z - dz)
            J_fd[:, :, k] = (x_plus - x_minus) / (2 * eps)

    jac_err = torch.max(torch.abs(J_exact - J_fd)).item()

    analytic = model.log_det_J(z)
    with torch.no_grad():
        sign_fd, logabs_fd = torch.linalg.slogdet(J_fd)
        fd = logabs_fd + 1j * torch.angle(sign_fd)
    det_err = torch.max(torch.abs(torch.exp(analytic) - torch.exp(fd))).item()

    tag = "rho+i*theta" if allow_mag else "i*theta only"
    print(f"  [{tag:12s}] max |J-J_fd| = {jac_err:.3e}   "
          f"max |detJ-detJ_fd| = {det_err:.3e}")
    if jac_err > tol or det_err > tol:
        raise AssertionError("Jacobian preflight failed -- do not trust log det J")


def exact_lattice_harmonic(N_val=N, a_c_val=a_c, m_val=m, omega_val=omega):
    """Closed-form zero-endpoint harmonic propagator on the N-site lattice (lambda=0)."""
    D_val = N_val - 1
    H = np.zeros((D_val, D_val), dtype=np.complex128)
    np.fill_diagonal(H, 2 * m_val / a_c_val - a_c_val * m_val * omega_val**2)
    np.fill_diagonal(H[1:], -m_val / a_c_val)
    np.fill_diagonal(H[:, 1:], -m_val / a_c_val)
    prefactor = (m_val / (2 * np.pi * 1j * a_c_val)) ** (N_val / 2)
    return prefactor * (2 * np.pi * 1j) ** (D_val / 2) / np.sqrt(np.linalg.det(H))


def mehler_kernel(x_i, x_f, T_val, m_val, omega_val, eta_val=0.0):
    """Exact continuum HO propagator at complex time T_c = T - i*eta."""
    wT = omega_val * (T_val - 1j * eta_val)
    prefactor = np.sqrt(m_val * omega_val / (2 * np.pi * 1j * np.sin(wT)))
    exponent = (1j * m_val * omega_val / (2 * np.sin(wT))) * (
        (x_i**2 + x_f**2) * np.cos(wT) - 2 * x_i * x_f)
    return prefactor * np.exp(exponent)


def relative_component_errors(K, ref, allow_sign_flip=ALLOW_GLOBAL_SIGN_FLIP):
    """Per-component relative error, optionally taking the better of +K and -K.

    The overall sign of K is set by a determinant branch, so a globally sign-flipped result
    is the same contour and is accepted (this was verified on the exact thimble, which
    returns exactly -reference).
    """
    def err(val):
        return (abs(val.real - ref.real) / max(abs(ref.real), 1e-15),
                abs(val.imag - ref.imag) / max(abs(ref.imag), 1e-15))
    e_plus = err(K)
    if not allow_sign_flip:
        return e_plus, +1
    e_minus = err(-K)
    return (e_minus, -1) if max(e_minus) < max(e_plus) else (e_plus, +1)


def validate_estimator(n_mc=40_000, tol=1e-4):
    """Feed the estimator the EXACT lambda=0 linear thimble x = A z, A = Q diag(sqrt(i/d_k)).

    This isolates the estimator from the training: if K comes back at the reference with
    ESS = 1, then complex_action, the (m/2 pi i a_c)^(N/2) normalisation and the log W
    assembly are all correct, and any later disagreement is the fault of the contour.
    """
    Q_h, d_h = hessian_modes(N, a_c, m, omega)
    A_np = Q_h @ np.diag(np.sqrt(1j / d_h))

    H = np.zeros((D, D), dtype=np.complex128)
    np.fill_diagonal(H, 2 * m / a_c - a_c * m * omega**2)
    np.fill_diagonal(H[1:], -m / a_c)
    np.fill_diagonal(H[:, 1:], -m / a_c)
    resid = np.abs(A_np.T @ H @ A_np - 1j * np.eye(D)).max()

    torch.manual_seed(1)
    z = torch.randn(n_mc, D)
    with torch.no_grad():
        x = z.to(torch.complex128) @ torch.tensor(A_np, dtype=torch.complex128).T
        S = complex_action(x, x_i_prop, x_f_prop, 0.0, a_c, m, omega)
        sgn, logabs = np.linalg.slogdet(A_np)
        log_det_J = complex(logabs + 1j * np.angle(sgn))
        log_P = -0.5 * torch.sum(z**2, dim=1) - (D / 2) * np.log(2 * np.pi)
        log_W = 1j * S + log_det_J - log_P.to(torch.complex128)

        shift = torch.max(log_W.real)
        W = torch.exp(log_W - shift)
        K = (m / (2 * np.pi * 1j * a_c)) ** (N / 2) * np.exp(shift.item()) * torch.mean(W).numpy()
        Wv = (m / (2 * np.pi * 1j * a_c)) ** (N / 2) * np.exp(shift.item()) * W.numpy()
        ess = float(np.abs(np.mean(Wv))**2 / np.mean(np.abs(Wv)**2))

    ref = LATTICE_REFERENCE_N32[0.0]
    (e_re, e_im), sign_used = relative_component_errors(K, ref)
    print(f"  A^T H A - iI  : {resid:.3e}")
    print(f"  K (exact thimble) = {K:.10f}   [{'+K' if sign_used > 0 else '-K'}]")
    print(f"  N=32 reference    = {ref:.10f}")
    print(f"  errors = ({100*e_re:.5f}%, {100*e_im:.5f}%)   ESS/N = {ess:.6f}")
    print(f"  Re S: mean={S.real.mean():.2e} std={S.real.std():.2e}   "
          f"Im S: mean={S.imag.mean():.3f}  (thimble target |z|^2/2 = {D/2:.1f})")
    print(f"  std(Re log W) = {log_W.real.std().item():.3e}   (0 = perfect importance sampler)")
    if max(e_re, e_im) > tol:
        raise AssertionError("estimator validation failed -- the MC assembly is wrong")
    print("  estimator validation PASSED (global sign is the known det branch ambiguity)")
    return K, ess


print("-- Jacobian vs finite differences --")
finite_difference_check()
print("Jacobian preflight PASSED")

print("\n-- lambda=0 references --")
K0_closed = exact_lattice_harmonic()
K0_table = LATTICE_REFERENCE_N32[0.0]
K_meh = mehler_kernel(x_i_prop, x_f_prop, T, m, omega, eta)
print(f"closed-form N=32 lattice : {K0_closed:.10f}")
print(f"tabulated N=32 reference : {K0_table:.10f}   (diff {abs(K0_closed - K0_table):.3e})")
print(f"continuum Mehler         : {K_meh:.10f}   <- separate discretisation limit")

print("\n-- estimator validation on the exact thimble --")
validate_estimator()

-- Jacobian vs finite differences --
  [i*theta only] max |J-J_fd| = 4.883e-10   max |detJ-detJ_fd| = 1.275e-09
  [rho+i*theta ] max |J-J_fd| = 8.160e-10   max |detJ-detJ_fd| = 4.777e-09
Jacobian preflight PASSED

-- lambda=0 references --
closed-form N=32 lattice : 0.3112704605-0.3012802812j
tabulated N=32 reference : 0.3121159000-0.3022604000j   (diff 1.294e-03)
continuum Mehler         : 0.3121370206-0.3022848712j   <- separate discretisation limit

-- estimator validation on the exact thimble --
  A^T H A - iI  : 7.022e-16
  K (exact thimble) = 0.3112704605-0.3012802812j   [+K]
  N=32 reference    = 0.3121159000-0.3022604000j
  errors = (0.27087%, 0.32426%)   ESS/N = 1.000000
  Re S: mean=-6.40e-17 std=3.48e-16   Im S: mean=1.996  (thimble target |z|^2/2 = 2.0)
  std(Re log W) = 6.756e-16   (0 = perfect importance sampler)


AssertionError: estimator validation failed -- the MC assembly is wrong

## 6. Loss, training and estimation

The loss is the scaled phase condition plus the descent condition, and nothing else. It sees
only the action — no $\log\det J$, no $\log P$, no variance of the weights:

```python
scale        = torch.abs(S).clamp_min(1e-12)
phase_loss   = torch.mean((S.real / scale)**2)
descent_loss = torch.mean(torch.relu(-S.imag)**2)
loss         = phase_loss + descent_loss
```

`scale` is *not* detached, so gradients flow through it.

Note on scales: `phase_loss` is dimensionless and $O(1)$, while `descent_loss` carries units
of $S^2$ and can reach $O(10^6)$ if `Im S` goes negative. That is the form as specified, so it
is kept, but the two terms are logged and plotted separately so any imbalance is visible.

Estimation is the standard self-normalised importance-sampling step, unchanged:

$$\log W = iS + \log\det J - \log P,\qquad
K = \Bigl(\frac{m}{2\pi i a_c}\Bigr)^{N/2}\mathbb{E}_{z\sim P}\bigl[e^{\log W}\bigr]$$

In [ ]:
def thimble_loss(S):
    """Scaled phase loss + descent loss. Gradients flow through `scale`.

    Weight is exp(iS): constant phase means Re(S) = const = 0 at the x=0 saddle, and
    descent means Re(iS) = -Im(S) <= 0, i.e. Im(S) >= 0.
    """
    scale = torch.abs(S).clamp_min(1e-12)
    phase_loss = torch.mean((S.real / scale) ** 2)
    descent_loss = torch.mean(torch.relu(-S.imag) ** 2)
    return phase_loss + descent_loss, phase_loss, descent_loss


def run_single(lam_val, seed=52, epochs_val=EPOCHS, batch_val=BATCH,
               N_mc_val=N_MC, lr=LR, verbose=True):
    """Train a rotation contour from scratch and estimate K(x_f, x_i; T_c)."""
    torch.manual_seed(seed)
    np.random.seed(seed)

    model = ThimbleRotor(D, Q_mode, s_mode, width=WIDTH, out_scale=OUT_SCALE,
                         allow_magnitude=ALLOW_MAGNITUDE)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    hist = {"total": [], "phase": [], "descent": [], "abs_S": [], "im_S": []}
    t0 = time.time()

    for epoch in range(epochs_val):
        optimizer.zero_grad()
        z = torch.randn(batch_val, D)                       # fresh standard normals
        x, _ = model(z)
        S = complex_action(x, x_i_prop, x_f_prop, lam_val, a_c, m, omega)
        loss, phase_loss, descent_loss = thimble_loss(S)

        if not torch.isfinite(loss):
            raise FloatingPointError(f"non-finite loss at epoch {epoch + 1}")
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=CLIP_GRAD)
        optimizer.step()

        hist["total"].append(loss.item())
        hist["phase"].append(phase_loss.item())
        hist["descent"].append(descent_loss.item())
        # |S| and Im S are tracked because phase_loss is scale invariant: driving |S| to 0
        # (collapse onto the saddle) or to infinity both make it small for the wrong reason.
        # On the exact thimble Im S = |z|^2/2 = D/2.
        with torch.no_grad():
            hist["abs_S"].append(torch.abs(S).mean().item())
            hist["im_S"].append(S.imag.mean().item())

        if verbose and (epoch == 0 or (epoch + 1) % LOG_EVERY == 0):
            print(f"    epoch {epoch+1:6d}/{epochs_val} | total={loss.item():.4e} | "
                  f"phase={phase_loss.item():.4e} | descent={descent_loss.item():.4e} | "
                  f"<|S|>={hist['abs_S'][-1]:.3e} <Im S>={hist['im_S'][-1]:.3e} | "
                  f"{time.time()-t0:.0f}s", flush=True)

    elapsed = time.time() - t0

    # ── Monte-Carlo estimate: log W = iS + log det J - log P ──
    norm_factor = (m / (2 * np.pi * 1j * a_c)) ** (N / 2)
    log_W_chunks = []
    remaining = N_mc_val
    while remaining:
        n_now = min(MC_CHUNK, remaining)
        z = torch.randn(n_now, D)
        log_det_J = model.log_det_J(z)                       # full complex Jacobian
        with torch.no_grad():
            x, _ = model(z)
            S = complex_action(x, x_i_prop, x_f_prop, lam_val, a_c, m, omega)
            log_P = -0.5 * torch.sum(z**2, dim=1) - (D / 2) * np.log(2 * np.pi)
            log_W_chunks.append((1j * S + log_det_J - log_P.to(torch.complex128)).cpu())
        remaining -= n_now

    log_W = torch.cat(log_W_chunks)
    shift = torch.max(log_W.real)
    W_shifted = torch.exp(log_W - shift)
    scale_mc = norm_factor * np.exp(shift.item())
    K_mc = scale_mc * torch.mean(W_shifted).numpy()
    W_vals = scale_mc * W_shifted.numpy()
    K_std = np.std(W_vals) / np.sqrt(N_mc_val)
    ess = float(np.abs(np.mean(W_vals))**2 / np.mean(np.abs(W_vals)**2))

    # contour-quality diagnostics: std(Re log W) is what the old variance loss minimised,
    # so it says whether this contour is a usable importance sampler at all (0 = perfect).
    std_re_logW = float(log_W.real.std())
    std_im_logW = float(log_W.imag.std())

    z_diag = torch.randn(256, D)
    sigma_min = torch.linalg.svdvals(model.jacobian_matrix(z_diag))[:, -1].cpu().numpy()

    lattice_ref = LATTICE_REFERENCE_N32.get(lam_val)
    (err_re, err_im), sign_used = (
        relative_component_errors(K_mc, lattice_ref) if lattice_ref is not None
        else ((np.nan, np.nan), 1))

    result = dict(
        lam=lam_val, seed=seed,
        K=complex(K_mc), K_re=float(K_mc.real), K_im=float(K_mc.imag),
        abs_K=float(abs(K_mc)), std_err=float(K_std), ess_n=ess,
        final_total=hist["total"][-1], final_phase=hist["phase"][-1],
        final_descent=hist["descent"][-1],
        hist=hist, min_sigma_J=float(np.min(sigma_min)),
        std_re_logW=std_re_logW, std_im_logW=std_im_logW,
        final_abs_S=hist["abs_S"][-1], final_im_S=hist["im_S"][-1],
        lattice_reference=lattice_ref,
        continuum_reference=CONTINUUM_REFERENCE.get(lam_val),
        err_re=err_re, err_im=err_im, sign_used=sign_used,
        time_s=elapsed,
        state_dict={k: v.detach().clone() for k, v in model.state_dict().items()},
    )

    if verbose:
        print(f"  => K = {K_mc:.10f}  |K| = {abs(K_mc):.8f}  stderr = {K_std:.2e}  "
              f"ESS/N = {ess:.5f}")
        print(f"     phase = {hist['phase'][-1]:.3e}  descent = {hist['descent'][-1]:.3e}  "
              f"min sigma(J) = {result['min_sigma_J']:.3e}  ({elapsed:.0f}s)")
        print(f"     std(Re log W) = {std_re_logW:.3e}  <|S|> = {hist['abs_S'][-1]:.3e}  "
              f"<Im S> = {hist['im_S'][-1]:.3e}  (thimble wants Im S = {D/2:.1f})")
        if lattice_ref is not None:
            tag = "+K" if sign_used > 0 else "-K"
            print(f"     N=32 lattice = {lattice_ref:.10f}  [{tag}]  "
                  f"Re err = {100*err_re:.3f}%  Im err = {100*err_im:.3f}%")
    return result


print("thimble_loss and run_single defined.")

## 7. Run — $\lambda = 0$ benchmark, then $\lambda = 0.7$ and $\lambda = 2.4$

One press runs every seed at $\lambda = 0$ first and reports PASS/FAIL against the exact
$N=32$ lattice Gaussian, then the anharmonic values. Set `STOP_ON_LAMBDA0_FAIL = True` in the
parameter cell to abort before the anharmonic runs instead of merely warning.

Per run this writes a two-panel loss curve (phase and descent **separately**) to `run_dir`,
appends a row to the CSV, and stores the weights.

In [ ]:
csv_path = os.path.join(run_dir, f"{run_tag}_propagator_results.csv")
csv_header = ["seed", "lam", "K_re", "K_im", "abs_K", "abs_K_ref", "std_err", "ESS_over_N",
              "phase_loss", "descent_loss", "total_loss", "min_sigma_J",
              "std_re_logW", "std_im_logW", "mean_abs_S", "mean_im_S", "thimble_im_S",
              "lattice_ref_re", "lattice_ref_im", "err_re_pct", "err_im_pct", "sign_used",
              "continuum_ref_re", "continuum_ref_im",
              "N", "T", "eta", "m", "omega", "epochs", "batch", "lr", "time_s"]
with open(csv_path, "w", newline="", encoding="utf-8") as f:
    csv.writer(f).writerow(csv_header)


def plot_loss_curves(res):
    """Phase loss and descent loss on separate panels, plus the action-scale diagnostic."""
    fig, axes = plt.subplots(1, 3, figsize=(18, 4))
    for ax, key, title in [
        (axes[0], "phase",   r"Phase loss  mean$[(\mathrm{Re}\,S/|S|)^2]$"),
        (axes[1], "descent", r"Descent loss  mean$[\mathrm{relu}(-\mathrm{Im}\,S)^2]$"),
    ]:
        series = np.asarray(res["hist"][key])
        positive = series[series > 0]
        floor = positive.min() * 0.5 if positive.size else 1e-16
        ax.semilogy(np.clip(series, floor, None), lw=0.8)
        ax.set_xlabel("Epoch")
        ax.set_ylabel("Loss")
        ax.set_title(title)
        ax.grid(True, alpha=0.3)
    if not np.any(np.asarray(res["hist"]["descent"]) > 0):
        axes[1].text(0.5, 0.5, "never activated\n(Im S > 0 throughout)",
                     transform=axes[1].transAxes, ha="center", va="center", fontsize=11,
                     bbox=dict(boxstyle="round", fc="lightyellow", ec="grey"))

    # phase_loss is scale invariant, so watch where |S| goes: collapse to 0 or blow-up to
    # infinity both drive the loss down for the wrong reason. Exact thimble: Im S = D/2.
    axes[2].plot(res["hist"]["abs_S"], lw=0.8, label=r"$\langle |S| \rangle$")
    axes[2].plot(res["hist"]["im_S"], lw=0.8, label=r"$\langle \mathrm{Im}\,S \rangle$")
    axes[2].axhline(D / 2, color="k", ls="--", lw=1.2,
                    label=rf"thimble $|z|^2/2 = {D/2:.1f}$")
    axes[2].set_xlabel("Epoch")
    axes[2].set_ylabel("Action scale")
    axes[2].set_title("Action scale (degeneracy check)")
    axes[2].legend(fontsize=8)
    axes[2].grid(True, alpha=0.3)

    fig.suptitle(f"seed={res['seed']}, lambda={res['lam']}, T_c={T}-{eta}i", fontsize=12)
    plt.tight_layout()
    fname = os.path.join(run_dir, f"loss_seed{res['seed']}_lam{res['lam']}.png")
    plt.savefig(fname, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"  saved {fname}")


all_results = {}
lambda0_failures = []

for lam in lambda_list:                      # lambda = 0 first, then anharmonic
    for seed in seed_list:
        print("\n" + "#" * 78)
        print(f"#  lambda = {lam}   |   seed {seed}")
        print("#" * 78, flush=True)

        res = run_single(lam, seed=seed)
        all_results[(seed, lam)] = res

        with open(csv_path, "a", newline="", encoding="utf-8") as f:
            cont = res["continuum_reference"]
            lat = res["lattice_reference"]
            csv.writer(f).writerow([
                seed, f"{lam:.6g}",
                f"{res['K_re']:.16e}", f"{res['K_im']:.16e}", f"{res['abs_K']:.16e}",
                "" if lat is None else f"{abs(lat):.16e}",
                f"{res['std_err']:.6e}", f"{res['ess_n']:.6e}",
                f"{res['final_phase']:.6e}", f"{res['final_descent']:.6e}",
                f"{res['final_total']:.6e}", f"{res['min_sigma_J']:.6e}",
                f"{res['std_re_logW']:.6e}", f"{res['std_im_logW']:.6e}",
                f"{res['final_abs_S']:.6e}", f"{res['final_im_S']:.6e}", f"{D/2:.6e}",
                "" if lat is None else f"{lat.real:.16e}",
                "" if lat is None else f"{lat.imag:.16e}",
                f"{100*res['err_re']:.6f}", f"{100*res['err_im']:.6f}", res["sign_used"],
                "" if cont is None else f"{cont.real:.16e}",
                "" if cont is None else f"{cont.imag:.16e}",
                N, T, eta, m, omega, EPOCHS, BATCH, LR, f"{res['time_s']:.1f}"])

        torch.save(res["state_dict"],
                   os.path.join(run_dir, f"weights_seed{seed}_lam{lam}.pt"))
        plot_loss_curves(res)

        if lam == 0.0:
            passed = max(res["err_re"], res["err_im"]) <= LAMBDA0_TOL
            print(f"  >>> lambda=0 benchmark {'PASS' if passed else 'FAIL'} "
                  f"(tol {100*LAMBDA0_TOL:.1f}% per component)")
            if not passed:
                lambda0_failures.append(seed)

    if lam == 0.0 and lambda0_failures:
        msg = (f"lambda=0 benchmark failed for seeds {lambda0_failures}. "
               "Anharmonic results are not trustworthy.")
        if STOP_ON_LAMBDA0_FAIL:
            raise RuntimeError(msg)
        print("\n" + "!" * 78)
        print("WARNING: " + msg)
        print("Continuing anyway because STOP_ON_LAMBDA0_FAIL is False.")
        print("!" * 78, flush=True)

print(f"\nCSV written to {csv_path}")

## 8. Summary — complex propagator values written to `runs`

Writes `results_summary.md` (human-readable, $K$ as a complex number per run) and
`results_summary.json` (machine-readable) into the run directory alongside the CSV.

The **seed-agreement** block is the important diagnostic. All seeds start from independent
random contours, so if the phase-plus-descent loss genuinely pins down the contour they must
agree on $K$. Wide scatter, or seeds with incompatible global phase, means the loss is
degenerate — and that is a result in itself, obtained in minutes rather than inferred later.

In [ ]:
json_rows, md = [], []
md.append(f"# Propagator results — {run_tag}\n")
md.append(f"Run: `{run_dir}`  \nGenerated: {datetime.now().isoformat(timespec='seconds')}\n")
_model_str = ("x = Q[(s*z) * exp(rho(z) + i theta(z))]" if ALLOW_MAGNITUDE
              else "x = Q[(s*z) * exp(i theta(z))]  (pure rotation)")
md.append(f"Model: `{_model_str}`, loss = scaled phase + descent, "
          f"Adam lr={LR}, batch={BATCH}, epochs={EPOCHS}\n")
md.append(f"Exact thimble diagnostic: on the true contour `<Im S> = |z|^2/2 = {D/2:.1f}`\n")
md.append(f"Complex time `T_c = {T} - {eta}i`, `N = {N}`, `D = {D}`, "
          f"endpoints `({x_i_prop}, {x_f_prop})`\n")

for lam in lambda_list:
    ref = LATTICE_REFERENCE_N32.get(lam)
    cont = CONTINUUM_REFERENCE.get(lam)
    md.append(f"\n## lambda = {lam}\n")
    md.append(f"- N=32 lattice reference: `{ref}`")
    md.append(f"- continuum reference:    `{cont}`\n")
    md.append(f"- exact \\|K\\|: `{abs(ref):.8f}`\n")
    md.append("| seed | K (complex) | \\|K\\| | ESS/N | phase | descent | <Im S> | Re err % | Im err % | sign |")
    md.append("|---|---|---|---|---|---|---|---|---|---|")

    for seed in seed_list:
        r = all_results[(seed, lam)]
        md.append(f"| {seed} | `{r['K']:.10f}` | {r['abs_K']:.8f} | {r['ess_n']:.2e} | "
                  f"{r['final_phase']:.2e} | {r['final_descent']:.2e} | "
                  f"{r['final_im_S']:.3f} | "
                  f"{100*r['err_re']:.3f} | {100*r['err_im']:.3f} | "
                  f"{'+K' if r['sign_used'] > 0 else '-K'} |")
        json_rows.append({
            "seed": seed, "lambda": lam,
            "K_complex_str": str(r["K"]), "K_re": r["K_re"], "K_im": r["K_im"],
            "abs_K": r["abs_K"], "std_err": r["std_err"], "ESS_over_N": r["ess_n"],
            "phase_loss": r["final_phase"], "descent_loss": r["final_descent"],
            "total_loss": r["final_total"], "min_sigma_J": r["min_sigma_J"],
            "std_re_logW": r["std_re_logW"], "mean_abs_S": r["final_abs_S"],
            "mean_im_S": r["final_im_S"], "thimble_im_S": D / 2,
            "abs_K_ref": None if ref is None else abs(ref),
            "lattice_ref_re": None if ref is None else ref.real,
            "lattice_ref_im": None if ref is None else ref.imag,
            "err_re_pct": 100 * r["err_re"], "err_im_pct": 100 * r["err_im"],
            "sign_used": r["sign_used"], "time_s": r["time_s"],
        })

    # seed agreement: independent random starts must land on the same K
    vals = np.array([all_results[(s, lam)]["K"] for s in seed_list], dtype=np.complex128)
    scatter = float(np.std(vals))
    rel_scatter = scatter / max(float(np.mean(np.abs(vals))), 1e-300)
    phases = np.angle(vals / vals[0])
    incompatible = bool(np.any(np.abs(phases) > np.pi / 2))
    md.append(f"\nSeed scatter `|std(K)| = {scatter:.4e}` "
              f"({100*rel_scatter:.2f}% of mean |K|)"
              + ("  \n**WARNING: seeds have incompatible global phase — do not average.**"
                 if incompatible else ""))

summary_md = os.path.join(run_dir, "results_summary.md")
with open(summary_md, "w", encoding="utf-8") as f:
    f.write("\n".join(md) + "\n")
with open(os.path.join(run_dir, "results_summary.json"), "w", encoding="utf-8") as f:
    json.dump(json_rows, f, indent=2)

print("=" * 100)
print("FINAL SUMMARY  (fixed-N lattice comparison first; global sign flip tolerated)")
print("=" * 100)
print(f"{'seed':>5s} {'lam':>6s} {'K_re':>14s} {'K_im':>14s} {'|K|':>10s} {'ESS/N':>9s} "
      f"{'phase':>10s} {'descent':>10s} {'Re err%':>9s} {'Im err%':>9s}")
print("-" * 100)
for lam in lambda_list:
    for seed in seed_list:
        r = all_results[(seed, lam)]
        print(f"{seed:5d} {lam:6.2f} {r['K_re']:14.8f} {r['K_im']:14.8f} "
              f"{r['abs_K']:10.6f} {r['ess_n']:9.2e} "
              f"{r['final_phase']:10.2e} {r['final_descent']:10.2e} "
              f"{100*r['err_re']:9.3f} {100*r['err_im']:9.3f}")
    ref = LATTICE_REFERENCE_N32.get(lam)
    vals = np.array([all_results[(s, lam)]["K"] for s in seed_list], dtype=np.complex128)
    print(f"{'':5s} {'ref':>6s} {ref.real:14.8f} {ref.imag:14.8f} {abs(ref):10.6f}"
          f"   seed scatter |std(K)| = {np.std(vals):.3e}")
    print("-" * 100)

# |K| is the clearest tell: the exact |K| falls steeply with lambda. A contour family that
# rotates the lambda dependence away instead of following the thimble gives a nearly
# lambda-independent |K| while still reporting tiny loss and ESS ~ 1.
print("\n|K| vs lambda  (exact |K| drops steeply; a flat ML column means the contour is "
      "lambda-blind)")
print(f"{'lam':>6s} {'|K| exact':>12s} {'|K| ML (mean)':>15s} {'ratio':>8s}")
for lam in lambda_list:
    ref = LATTICE_REFERENCE_N32.get(lam)
    ml = float(np.mean([all_results[(s, lam)]["abs_K"] for s in seed_list]))
    print(f"{lam:6.2f} {abs(ref):12.6f} {ml:15.6f} {ml/abs(ref):8.3f}")

print(f"\nSaved: {csv_path}")
print(f"Saved: {summary_md}")
print(f"Saved: {os.path.join(run_dir, 'results_summary.json')}")

## 9. Combined plots

Left: $K$ against $\lambda$ per seed, with the exact $N=32$ lattice references. A curve that
tracks the references is the goal; a curve that stays flat as $\lambda$ grows is the failure
mode this rewrite is meant to break.

Right: both loss components for every run on one axis, so a run whose descent term never
activates (or never settles) is obvious.

In [ ]:
colors = plt.cm.tab10.colors
lams = sorted(lambda_list)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for s_idx, seed in enumerate(seed_list):
    c = colors[s_idx % len(colors)]
    axes[0].plot(lams, [all_results[(seed, l)]["K_re"] for l in lams],
                 "o-", color=c, ms=5, label=f"seed {seed}  Re K")
    axes[0].plot(lams, [all_results[(seed, l)]["K_im"] for l in lams],
                 "s--", color=c, ms=5, alpha=0.7, label=f"seed {seed}  Im K")
    for l in lams:
        h = all_results[(seed, l)]["hist"]
        axes[1].semilogy(np.clip(h["phase"], 1e-16, None), color=c, alpha=0.75, lw=0.7,
                         label=f"seed {seed} phase" if l == lams[0] else None)
        axes[1].semilogy(np.clip(h["descent"], 1e-16, None), color=c, alpha=0.35, lw=0.7,
                         ls=":", label=f"seed {seed} descent" if l == lams[0] else None)

axes[0].plot(lams, [LATTICE_REFERENCE_N32[l].real for l in lams],
             "k*-", ms=13, lw=1.2, label="exact lattice  Re K")
axes[0].plot(lams, [LATTICE_REFERENCE_N32[l].imag for l in lams],
             "kP--", ms=10, lw=1.2, label="exact lattice  Im K")
axes[0].axhline(0, color="grey", lw=0.5)
axes[0].set_xlabel(r"$\lambda$")
axes[0].set_ylabel(r"$K$")
axes[0].set_title(rf"$K$ vs $\lambda$   ($T_c={T}-{eta}i$, $N={N}$)")
axes[0].legend(fontsize=7, ncol=2)
axes[0].grid(True, alpha=0.3)

axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Loss")
axes[1].set_title("Phase (solid) and descent (dotted), all runs")
axes[1].legend(fontsize=7, ncol=2)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
fname = os.path.join(run_dir, "summary_K_vs_lambda.png")
plt.savefig(fname, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved {fname}")